# B3 amended — candidate mapping for every validation/test target

This notebook calls the frozen `b3-verified-patch-v1` implementation. It loads all Devign and REVEAL validation/test targets, ranks **source-backed but unreviewed** pair candidates for human review, and (when a genuinely dual-reviewed pair pool is supplied) calls `VerifiedPairSelector` to choose the registered best pair for every target.

It does not run model inference, alter thresholds/specs, or treat heuristic name matches as verified patch pairs. The test target label and historical demo fields are discarded before ranking. Without authentic source evidence, it emits an explicit paused row for every target rather than inventing pairs.

**Required inputs:** repository freeze at `01_Author_Baseline_Unified/output/research_freeze_20261101`, frozen specs archive `grace_15arms_prompts_and_clean_specs.zip`, and a JSONL source-pair evidence file described below. Run locally from the repository root or set `REPO_ROOT_OVERRIDE`.

In [1]:
from pathlib import Path
import hashlib, json, zipfile
from collections import Counter
from fractions import Fraction
from datetime import datetime, timezone

# Set this if the notebook server's working directory is outside the repository.
REPO_ROOT_OVERRIDE = None
if REPO_ROOT_OVERRIDE:
    ROOT = Path(REPO_ROOT_OVERRIDE).resolve()
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "01_Author_Baseline_Unified").is_dir():
        if ROOT.parent == ROOT:
            raise FileNotFoundError("Set REPO_ROOT_OVERRIDE to the GRACE-summary repository root")
        ROOT = ROOT.parent

ARM_DIR = ROOT / "13_Arm_B3_Contrastive"
FREEZE = ROOT / "01_Author_Baseline_Unified/output/research_freeze_20261101"
SPECS_ZIP = ROOT / "grace_15arms_prompts_and_clean_specs.zip"
PAIR_EVIDENCE = ARM_DIR / "inputs/source_pair_evidence.jsonl"
TOP_K_REVIEW_CANDIDATES = 3

EXPECTED_FREEZE_MANIFEST_SHA256 = "e1d3c6b17c3a0b5c1e05203794d48a33825e0ea00a55deaaef565af25efe0448"
EXPECTED_SPECS_ZIP_SHA256 = "9db0d3eebb66a2f83c1f9803ceb0192954a3dfe2c3de9515188a996ede84ed7b"
DATASETS = ("devign", "reveal")
SPLITS = ("validation", "test")

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""): h.update(chunk)
    return h.hexdigest()
def jsonl_bytes(data):
    return [json.loads(line) for line in data.decode("utf-8").splitlines() if line.strip()]
def jsonl_file(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as f:
        for row in rows: f.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")

SRC = ROOT / "01_Author_Baseline_Unified/diagnostic_kit/src"
import sys
sys.path.insert(0, str(SRC))
from grace_diag.amended_b3 import (
    VerifiedPairSelector, make_pair_candidate, payload_hash, validate_pair, digest, function_hint
)
from grace_diag.contrastive_pair_builder import normalize_code_tokens


## 1. Verify frozen inputs and load the entire census

Targets are formed from the registered validation/test specs after retaining only ID, code, and graph fields. Their labels, old demonstrations, and old explanations do not enter ranking. Donors come only from the frozen clean train bank. The notebook checks the repository freeze and spec-archive hashes before proceeding.

In [2]:
assert FREEZE.is_dir(), f"Missing frozen research bank: {FREEZE}"
assert SPECS_ZIP.is_file(), f"Missing frozen specs archive: {SPECS_ZIP}"
freeze_manifest_path = FREEZE / "freeze_manifest.json"
assert sha256_file(freeze_manifest_path) == EXPECTED_FREEZE_MANIFEST_SHA256, "Freeze manifest hash mismatch"
assert sha256_file(SPECS_ZIP) == EXPECTED_SPECS_ZIP_SHA256, "Frozen specs archive hash mismatch"
freeze_manifest = json.loads(freeze_manifest_path.read_text(encoding="utf-8"))

def load_frozen_bank(dataset):
    ds_dir = FREEZE / dataset
    manifest = freeze_manifest["datasets"][dataset]
    for filename in ("train_raw.jsonl", "train_manifest.jsonl", "validation_raw.jsonl", "test_raw.jsonl"):
        p = ds_dir / filename
        assert p.is_file(), f"Missing frozen file: {p}"
        assert sha256_file(p) == manifest["files_sha256"][filename], f"Frozen file hash mismatch: {dataset}/{filename}"
    meta = {r["id"]: r for r in jsonl_file(ds_dir / "train_manifest.jsonl")}
    bank = {}
    for d in jsonl_file(ds_dir / "train_raw.jsonl"):
        m = meta[d["id"]]
        if not m["keep_in_bank"]: continue
        assert digest(d["func"]) == m["code_sha256"] and d["target"] == m["label"]
        assert d["split"] == "train" and d["dataset"] == dataset and d["id"] not in bank
        bank[d["id"]] = {**d, "function_uid": d["id"]}
    assert len(bank) == manifest["donor_count"], f"Donor census mismatch for {dataset}"
    return bank

banks = {ds: load_frozen_bank(ds) for ds in DATASETS}
with zipfile.ZipFile(SPECS_ZIP) as specs_zip:
    targets_by_dataset_split = {}
    target_projects = {}
    for ds in DATASETS:
        for split in SPLITS:
            spec_name = f"specs/{ds}_{split}_15arms.jsonl"
            rows = jsonl_bytes(specs_zip.read(spec_name))
            raw_map = {r["id"]: r for r in jsonl_file(FREEZE / ds / f"{split}_raw.jsonl")}
            targets = []
            for r in rows:
                # Whitelist only model input fields; ignore label/demo/explanation metadata.
                target = {k: r[k] for k in ("id", "query_code", "nodes", "edges") if k in r}
                assert {"id", "query_code", "nodes", "edges"} <= target.keys(), f"Incomplete target: {ds}/{split}/{r.get('id')}"
                raw = raw_map.get(target["id"])
                assert raw is not None and digest(raw["func"].strip()) == digest(target["query_code"].strip()), f"Frozen target/spec mismatch: {ds}/{split}/{target['id']}"
                target["dataset"] = ds
                target["split"] = split
                targets.append(target)
                target_projects[(ds, split, target["id"])] = raw.get("project")
            assert len({t["id"] for t in targets}) == len(targets), f"Duplicate IDs: {ds}/{split}"
            targets_by_dataset_split[(ds, split)] = targets

target_count = sum(map(len, targets_by_dataset_split.values()))
print({f"{ds}/{split}": len(targets_by_dataset_split[(ds, split)]) for ds in DATASETS for split in SPLITS})
print("Total validation + test targets:", target_count)

{'devign/validation': 2732, 'devign/test': 2732, 'reveal/validation': 2273, 'reveal/test': 2274}
Total validation + test targets: 10011


## 2. Generate per-target discovery candidates from the frozen donor bank

This first pass gives every validation and test target a coverage row. It proposes opposite-label donors sharing the target project and extracted function-name hint, then ranks those leads by mean token Jaccard. These are discovery leads only: same project/name does not establish a source patch relationship, so they are not eligible B3 pairs and cannot be rendered. The next section admits only candidates backed by actual repository patch evidence.


In [3]:
from collections import defaultdict

donor_groups = {ds: defaultdict(lambda: {1: [], 0: []}) for ds in DATASETS}
donor_token_sets = {}
for ds in DATASETS:
    for donor in banks[ds].values():
        hint = function_hint(donor["func"])
        project = donor.get("project")
        if hint and project is not None and donor["target"] in (0, 1):
            donor_groups[ds][(project, hint)][donor["target"]].append(donor)
            donor_token_sets[(ds, donor["id"])] = set(normalize_code_tokens(donor["func"]))

def score_pair_tokens(query_tokens, vulnerable, patched):
    def jaccard(tokens):
        return Fraction(len(query_tokens & tokens), len(query_tokens | tokens)) if query_tokens and tokens else Fraction(0)
    return (jaccard(donor_token_sets[(vulnerable["dataset"], vulnerable["id"])]) +
            jaccard(donor_token_sets[(patched["dataset"], patched["id"])]) ) / 2

discovery_candidates = []
discovery_coverage = []
for ds in DATASETS:
    for split in SPLITS:
        for target in targets_by_dataset_split[(ds, split)]:
            hint = function_hint(target["query_code"])
            project = target_projects.get((ds, split, target["id"]))
            group = donor_groups[ds].get((project, hint), {1: [], 0: []}) if hint and project is not None else {1: [], 0: []}
            query_tokens = set(normalize_code_tokens(target["query_code"]))
            ranked = []
            for vulnerable in group[1]:
                for patched in group[0]:
                    if vulnerable["id"] == patched["id"]:
                        continue
                    ranked.append((score_pair_tokens(query_tokens, vulnerable, patched), vulnerable, patched))
            ranked.sort(key=lambda x: (-x[0], x[1]["id"], x[2]["id"]))
            discovery_coverage.append({"dataset": ds, "split": split, "target_id": target["id"],
                "status": "HEURISTIC_CANDIDATES_FOUND" if ranked else "NO_PROJECT_FUNCTION_MATCH",
                "candidate_count": len(ranked)})
            for rank, (score, vulnerable, patched) in enumerate(ranked[:TOP_K_REVIEW_CANDIDATES], start=1):
                discovery_candidates.append({"dataset": ds, "split": split, "target_id": target["id"],
                    "rank": rank, "status": "UNVERIFIED_DISCOVERY_ONLY", "project": project, "function_hint": hint,
                    "vulnerable_id": vulnerable["id"], "patched_candidate_id": patched["id"],
                    "vulnerable_commit_id": vulnerable.get("commit_id"), "patched_candidate_commit_id": patched.get("commit_id"),
                    "vulnerable_sha256": digest(vulnerable["func"]), "patched_candidate_sha256": digest(patched["func"]),
                    "mean_jaccard_discovery_score": float(score),
                    "warning": "Project/function-name similarity is not evidence of a security patch. Do not use as a B3 pair."})

print("Targets covered:", len(discovery_coverage))
print("Targets with heuristic leads:", sum(row["candidate_count"] > 0 for row in discovery_coverage))


Targets covered: 10011
Targets with heuristic leads: 1220


## 3. Build source-backed pair candidates

Provide `13_Arm_B3_Contrastive/inputs/source_pair_evidence.jsonl`, one record per proposed pair, with this shape:

```json
{"dataset":"devign","pair_id":"stable-id","vulnerable_id":"devign:train:...","patched_id":"devign:train:...","source":{"repository":"owner/repo","path":"src/file.c","function":"function_name","fix_commit":"<40-char commit>","parent_commit":"<40-char parent>","source_url":"https://...","security_rationale":"...","before":"<exact vulnerable donor function>","after":"<exact patched donor function>","fix_diff":"<actual repository patch diff including --- a/path and +++ b/path>"},"reviews":[]}
```

The code checks evidence/donor consistency and constructs `diff` and `Key difference` from the actual before/after functions. Missing or inconsistent evidence is written to a rejection file. A candidate is **not verified** until two distinct reviewers independently approve the same evidence hash. Do not add synthetic approvals. The ranked pre-review list is only a review queue; official assignments use `VerifiedPairSelector` on the approved pool.

In [4]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = ARM_DIR / "output" / f"b3_amended_candidates_{run_id}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
write_jsonl(OUTPUT_DIR / "heuristic_discovery_candidates_all_targets.jsonl", discovery_candidates)
write_jsonl(OUTPUT_DIR / "heuristic_discovery_coverage_all_targets.jsonl", discovery_coverage)

pair_inputs = jsonl_file(PAIR_EVIDENCE) if PAIR_EVIDENCE.is_file() else []
candidate_pairs = {ds: [] for ds in DATASETS}
rejected_pair_inputs = []
seen_pair_ids = set()
for row_number, item in enumerate(pair_inputs, start=1):
    try:
        ds = item["dataset"]
        if ds not in DATASETS: raise ValueError("dataset must be devign or reveal")
        pair_id = item["pair_id"]
        if (ds, pair_id) in seen_pair_ids: raise ValueError("duplicate pair_id within dataset")
        vulnerable = banks[ds][item["vulnerable_id"]]
        patched = banks[ds][item["patched_id"]]
        pair = make_pair_candidate(ds, pair_id, vulnerable, patched, item["source"])
        pair["reviews"] = item.get("reviews", [])
        # Structural/evidence check always runs; human approval is checked separately below.
        validate_pair(pair, banks[ds], ds, require_reviews=False)
        seen_pair_ids.add((ds, pair_id))
        candidate_pairs[ds].append(pair)
    except Exception as exc:
        rejected_pair_inputs.append({"row_number": row_number, "pair_id": item.get("pair_id"), "error": f"{type(exc).__name__}: {exc}"})

write_jsonl(OUTPUT_DIR / "pair_candidates_for_review.jsonl", [p for ds in DATASETS for p in candidate_pairs[ds]])
write_jsonl(OUTPUT_DIR / "rejected_pair_inputs.jsonl", rejected_pair_inputs)
print("Authentic-source candidate records accepted structurally:", {ds: len(v) for ds, v in candidate_pairs.items()})
print("Rejected evidence rows:", len(rejected_pair_inputs))
print("Evidence input found:", PAIR_EVIDENCE.is_file())

Authentic-source candidate records accepted structurally: {'devign': 0, 'reveal': 0}
Rejected evidence rows: 0
Evidence input found: False


## 4. Rank candidate pairs for every validation and test target

The next output is a **human-review queue**, not an accepted prompt set. It ranks source-backed candidate records by the same mean token-set Jaccard used by amended B3, but marks all unapproved records as pending. Every target gets a status row, including targets with no eligible source evidence.

In [5]:
def mean_pair_jaccard(query_code, pair, bank):
    q = set(normalize_code_tokens(query_code))
    left = set(normalize_code_tokens(bank[pair["vulnerable_id"]]["func"]))
    right = set(normalize_code_tokens(bank[pair["patched_id"]]["func"]))
    def jaccard(other): return Fraction(len(q & other), len(q | other)) if q and other else Fraction(0)
    return (jaccard(left) + jaccard(right)) / 2

review_queue = []
all_targets_status = []
verified_assignments = []
for ds in DATASETS:
    pairs = candidate_pairs[ds]
    approved = []
    for pair in pairs:
        try:
            validate_pair(pair, banks[ds], ds, require_reviews=True)
            approved.append(pair)
        except ValueError:
            pass  # Expected for candidate records awaiting human review.
    selector = VerifiedPairSelector(approved, banks[ds], ds) if approved else None
    for split in SPLITS:
        for target in targets_by_dataset_split[(ds, split)]:
            ranked = sorted(
                ((mean_pair_jaccard(target["query_code"], pair, banks[ds]), pair) for pair in pairs),
                key=lambda item: (-item[0], item[1]["pair_id"])
            )
            if ranked:
                for rank, (score, pair) in enumerate(ranked[:TOP_K_REVIEW_CANDIDATES], start=1):
                    review_queue.append({
                        "dataset": ds, "split": split, "target_id": target["id"],
                        "rank_pre_review": rank, "pair_id": pair["pair_id"],
                        "vulnerable_id": pair["vulnerable_id"], "patched_id": pair["patched_id"],
                        "mean_jaccard": float(score), "pair_payload_sha256": payload_hash(pair),
                        "review_status": "PENDING_TWO_INDEPENDENT_REVIEWS",
                        "candidate_status": "UNREVIEWED_RANKING_ONLY"
                    })
                status = "CANDIDATES_RANKED_PENDING_REVIEW"
            else:
                status = "PAUSED_NO_SOURCE_EVIDENCE_CANDIDATES"
            all_targets_status.append({"dataset": ds, "split": split, "target_id": target["id"],
                                      "status": status, "candidate_count": min(len(ranked), TOP_K_REVIEW_CANDIDATES)})
            if selector is not None:
                selected_pair, demos, selection = selector.select(target["query_code"])
                verified_assignments.append({
                    "dataset": ds, "split": split, "target_id": target["id"],
                    "status": "VERIFIED_PAIR_SELECTED_PENDING_PROMPT_AUDIT",
                    "pair_id": selected_pair["pair_id"], "pair_payload_sha256": payload_hash(selected_pair),
                    "vulnerable_id": selected_pair["vulnerable_id"], "patched_id": selected_pair["patched_id"],
                    "selection": selection
                })
            else:
                verified_assignments.append({"dataset": ds, "split": split, "target_id": target["id"],
                                            "status": "PAUSED_NO_DUAL_REVIEWED_PAIR_POOL"})

write_jsonl(OUTPUT_DIR / "target_pair_review_queue_topk.jsonl", review_queue)
write_jsonl(OUTPUT_DIR / "target_status_all_validation_test.jsonl", all_targets_status)
write_jsonl(OUTPUT_DIR / "verified_pair_assignments_all_validation_test.jsonl", verified_assignments)
report = {
    "method": "b3-verified-patch-v1", "run_id_utc": run_id,
    "scope": "source-pair candidate ranking and verified pair assignment only; no inference",
    "target_labels_used": False, "historical_demos_used": False, "thresholds_changed": False,
    "datasets": {ds: {split: len(targets_by_dataset_split[(ds, split)]) for split in SPLITS} for ds in DATASETS},
    "target_total": target_count, "source_pair_records_input": len(pair_inputs),
    "source_pair_records_structurally_accepted": {ds: len(candidate_pairs[ds]) for ds in DATASETS},
    "source_pair_records_rejected": len(rejected_pair_inputs),
    "dual_reviewed_pairs": {ds: 0 for ds in DATASETS},
    "target_review_queue_rows": len(review_queue),
    "targets_with_pre_review_candidates": sum(r["candidate_count"] > 0 for r in all_targets_status),
    "targets_with_verified_assignment": sum(r["status"] == "VERIFIED_PAIR_SELECTED_PENDING_PROMPT_AUDIT" for r in verified_assignments),
    "targets_paused": sum(r["status"].startswith("PAUSED") for r in verified_assignments),
    "outputs": ["heuristic_discovery_candidates_all_targets.jsonl", "heuristic_discovery_coverage_all_targets.jsonl",
                "pair_candidates_for_review.jsonl", "rejected_pair_inputs.jsonl",
                "target_pair_review_queue_topk.jsonl", "target_status_all_validation_test.jsonl",
                "verified_pair_assignments_all_validation_test.jsonl"]
}
# Count reviewed records safely; invalid review rows remain unreviewed.
for ds in DATASETS:
    count = 0
    for p in candidate_pairs[ds]:
        try: validate_pair(p, banks[ds], ds, require_reviews=True); count += 1
        except ValueError: pass
    report["dual_reviewed_pairs"][ds] = count
(OUTPUT_DIR / "run_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("Output directory:", OUTPUT_DIR)

{
  "method": "b3-verified-patch-v1",
  "run_id_utc": "20261108T095236Z",
  "scope": "source-pair candidate ranking and verified pair assignment only; no inference",
  "target_labels_used": false,
  "historical_demos_used": false,
  "thresholds_changed": false,
  "datasets": {
    "devign": {
      "validation": 2732,
      "test": 2732
    },
    "reveal": {
      "validation": 2273,
      "test": 2274
    }
  },
  "target_total": 10011,
  "source_pair_records_input": 0,
  "source_pair_records_structurally_accepted": {
    "devign": 0,
    "reveal": 0
  },
  "source_pair_records_rejected": 0,
  "dual_reviewed_pairs": {
    "devign": 0,
    "reveal": 0
  },
  "target_review_queue_rows": 0,
  "targets_with_pre_review_candidates": 0,
  "targets_with_verified_assignment": 0,
  "targets_paused": 10011,
  "outputs": [
    "heuristic_discovery_candidates_all_targets.jsonl",
    "heuristic_discovery_coverage_all_targets.jsonl",
    "pair_candidates_for_review.jsonl",
    "rejected_pair_inputs

## Reading the outputs

- `pair_candidates_for_review.jsonl`: structurally checked unreviewed pair packets with actual-source-derived diff and explanation; reviewers must independently authenticate source revisions, same-function identity, security-fix relevance, and explanation support.
- `target_pair_review_queue_topk.jsonl`: top candidate pair suggestions for every target when evidence rows exist. These rankings prioritize review only; they are not official assignments while unreviewed.
- `verified_pair_assignments_all_validation_test.jsonl`: exact amended selector output for every validation/test target if a dual-reviewed pool exists; otherwise each target is explicitly paused.
- `target_status_all_validation_test.jsonl`: coverage row for every target regardless of candidate availability.

After approval, add the two review records to the corresponding pair record in `source_pair_evidence.jsonl`, rerun the notebook, inspect the manifest and prompt audit, and only then create/freeze B3-family prompts. A target-level selection file is not itself an inference-ready prompt bundle.